# 01 · Obtención de datos

Descarga la base de **buques grandes 2024** publicada en el Anexo V ("Flujos de tráfico 2022-2024") de la licitación de la Vía Navegable Troncal, en la página de la Agencia Nacional de Puertos y Navegación.

El archivo se guarda tal cual en `datos/crudo/` y **no se edita a mano**.

## Librerías

Antes de correr, elegir como kernel el Python del entorno `.venv` de esta carpeta. La primera celda muestra qué Python se está usando e instala en él las versiones de `requirements.txt` (si ya están instaladas, no hace nada).

In [1]:
import sys

print(sys.executable)  # debería terminar en .venv\Scripts\python.exe
%pip install -q -r requirements.txt

c:\Users\Damián\Damian - Facultad\Maestria en Economía\python\Damian_Tp_integrador\.venv\Scripts\python.exe
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from datetime import date
from pathlib import Path

import pandas as pd
import requests
from bs4 import BeautifulSoup

## Parámetros

In [3]:
URL_PAGINA = (
    "https://www.argentina.gob.ar/economia/agencia-nacional-de-puertos-y-navegacion/"
    "llamado-licitacion-publica-nacional-e"
)
TEXTO_LINK = "Buques grandes 2024"  # texto del botón en la página

CARPETA_CRUDO = Path("datos/crudo")  # ruta relativa a la carpeta del proyecto
CARPETA_CRUDO.mkdir(parents=True, exist_ok=True)

## Paso 1. Pedir el HTML de la página

In [ ]:
respuesta = requests.get(URL_PAGINA, timeout=30)
respuesta.raise_for_status()  # si el servidor responde con error, se corta acá
print(respuesta.status_code)

200


## Paso 2. Buscar el link del archivo dentro del HTML

In [ ]:
soup = BeautifulSoup(respuesta.text, "html.parser")

link = soup.find("a", string=TEXTO_LINK)
assert link is not None, f"No se encontró el link '{TEXTO_LINK}' en la página"

href = link["href"]
print(href)

blank:#https://www.argentina.gob.ar/sites/default/files/2025/12/2024_buques_grandes_protegido.xlsx


El `href` de la página viene con el prefijo `blank:#` antes de la dirección real (así lo arma el sitio para abrir el archivo en otra pestaña). Lo sacamos para quedarnos con la URL del archivo.

In [ ]:
url_archivo = href.replace("blank:#", "")
assert url_archivo.startswith("https://") and url_archivo.endswith(".xlsx")

nombre_archivo = url_archivo.split("/")[-1]
destino = CARPETA_CRUDO / nombre_archivo
print(url_archivo)
print(destino)

https://www.argentina.gob.ar/sites/default/files/2025/12/2024_buques_grandes_protegido.xlsx
datos\crudo\2024_buques_grandes_protegido.xlsx


## Paso 3. Descargar el archivo (solo si no está en disco)

In [ ]:
def descargar_archivo(url, destino):
    """
    Descarga el archivo de `url` y lo guarda en `destino`, sin modificarlo.

    Si el archivo ya existe, no lo vuelve a descargar.

    Input:
        url (str): dirección del archivo en la web
        destino (Path): ruta donde guardarlo
    Output:
        Path: la ruta del archivo en disco
    """
    if destino.exists():
        print(f"Ya existe, no se descarga de nuevo: {destino}")
        return destino

    respuesta = requests.get(url, timeout=120)
    respuesta.raise_for_status()
    destino.write_bytes(respuesta.content)  # bytes: el .xlsx es un archivo binario
    print(f"Descargado el {date.today()}: {destino} ({len(respuesta.content) / 1e6:.1f} MB)")
    return destino

In [ ]:
ruta_buques = descargar_archivo(url_archivo, destino)

Ya existe, no se descarga de nuevo: datos\crudo\2024_buques_grandes_protegido.xlsx


## Paso 4. Leer la base y darle un primer vistazo

In [ ]:
pd.ExcelFile(ruta_buques).sheet_names

['Original']

In [ ]:
buques = pd.read_excel(ruta_buques, sheet_name="Original")
print(buques.shape)
buques.head()

(18616, 28)


,informante,fecha,hora,seccion_o,origen,dpuerto,seccion_d,destino,hpuerto,matricula,...,agencia,tb,td,total,id_mov,imo,fact_iva,tipo_b_b,descrip,n_comp
0,RECALADA,2024-01-01,03:30:00,0.0,I. MARITIMO (Reca),NaN,1.1,ZONA COMUN,NaN,3E4937,...,FERTIMPORT S.A.,953.84,7717.65,8671.49,2413827,9250165.0,I,TB,TANQUE,FC A 0021-00025380
1,RECALADA,2024-01-01,03:33:00,1.1,ZONA COMUN,NaN,1.2,RIO PARANA GUAZU,NaN,3E4937,...,GOBIERNO,0.00,0.00,0.00,2417109,9250165.0,I,TB,TANQUE,NaN
2,RECALADA,2024-01-01,03:35:00,0.0,I. MARITIMO (Reca),NaN,1.2,DOCK SUD,NaN,LW4200,...,RAIZEN ARGENTINA S.A.U,3988.43,16297.73,20286.16,2413829,9253765.0,N,TB,TANQUE,FC A 0021-00028064
3,RECALADA,2024-01-01,03:36:00,1.2,RIO PARANA GUAZU,NaN,1.3,RPP KM. 460,NaN,3E4937,...,FERTIMPORT S.A.,716.29,6686.09,7402.38,2417110,9250165.0,I,TB,TANQUE,FC A 0021-00025380
4,RECALADA,2024-01-01,03:39:00,1.3,RPP KM. 460,NaN,2.0,RENOVA TIMBUES,NaN,3E4937,...,FERTIMPORT S.A.,127.91,489.91,617.82,2417004,9250165.0,I,TB,TANQUE,FC A 0021-00025408


In [ ]:
buques.dtypes

informante               str
fecha         datetime64[us]
hora                     str
seccion_o            float64
origen                   str
dpuerto                  str
seccion_d            float64
destino                  str
hpuerto                  str
matricula                str
buque                    str
mg                       str
trn                  float64
calado               float64
eslora               float64
manga                float64
bandera                  str
caladoreal           float64
agencia                  str
tb                   float64
td                   float64
total                float64
id_mov                 int64
imo                  float64
fact_iva                 str
tipo_b_b                 str
descrip                  str
n_comp                   str
dtype: object